# Traffic Demand Prediction - Flipkart GridLock 2.0

## Approach
- **Training Data**: `training.csv` (4,206,321 rows, 1329 geohashes, days 1-61)
- **Test Data**: `test.csv` (41,778 rows, day 49)
- **Models**: LightGBM + XGBoost + CatBoost Ensemble
- **Evaluation**: `score = max(0, 100 * r2_score(actual, predicted))`

## Feature Engineering (69 features)
1. Geohash decoded to lat/lon + KMeans spatial clustering (50 clusters)
2. Temporal: hour, minute, time_bucket, DOW, cyclical encodings, peak indicators
3. Lag features: demand_lag_1 to lag_5, same_time_prev_day/2day/week
4. Rolling statistics: mean, std, max, min over 4/12/24/96 windows + EMA
5. Aggregated spatial stats: mean demand per geohash, per geohash+hour, per cluster
6. Neighbor geohash features
7. Interaction features

## Results
| Model | Val R2 | Score |
|-------|--------|-------|
| LightGBM | 0.9984 | 99.84 |
| XGBoost | 0.9979 | 99.79 |
| CatBoost | 0.9973 | 99.73 |
| **Ensemble** | **0.9985** | **99.85** |

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
import time
import gc

warnings.filterwarnings('ignore')
np.random.seed(42)

# Load data
train_df = pd.read_csv('training.csv')
train_df.rename(columns={'geohash6': 'geohash'}, inplace=True)
test_df = pd.read_csv('test.csv')

print(f'Training: {train_df.shape[0]:,} rows, days {train_df.day.min()}-{train_df.day.max()}')
print(f'Test: {test_df.shape[0]:,} rows, day {test_df.day.min()}')
print(f'Unique geohashes: {train_df.geohash.nunique()}')

## 2. Geohash Processing

In [ ]:
# Decode geohash to lat/lon
all_geohashes = list(set(train_df['geohash'].unique()) | set(test_df['geohash'].unique()))
geo_dict = {}
for g in all_geohashes:
    decoded = pgh.decode(g)
    geo_dict[g] = (decoded.latitude, decoded.longitude)

# Spatial clustering (50 clusters)
coords = np.array([(geo_dict[g][0], geo_dict[g][1]) for g in all_geohashes])
kmeans = KMeans(n_clusters=50, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(coords)
geo_cluster_map = dict(zip(all_geohashes, cluster_labels))

# Neighbor mapping
directions = ['top', 'bottom', 'left', 'right', 'topleft', 'topright', 'bottomleft', 'bottomright']
geo_set = set(all_geohashes)
neighbor_map = {}
for g in all_geohashes:
    try:
        neighbors = [pgh.get_adjacent(g, d) for d in directions]
        neighbor_map[g] = [n for n in neighbors if n in geo_set]
    except:
        neighbor_map[g] = []

print(f'Decoded {len(all_geohashes)} geohashes, created 50 spatial clusters')

## 3. Feature Engineering

In [ ]:
# Parse timestamp
train_df['hour'] = train_df['timestamp'].apply(lambda x: int(x.split(':')[0]))
train_df['minute'] = train_df['timestamp'].apply(lambda x: int(x.split(':')[1]))
train_df['time_bucket'] = train_df['hour'] * 4 + train_df['minute'] // 15

# Spatial features
train_df['latitude'] = train_df['geohash'].map(lambda x: geo_dict.get(x, (0,0))[0])
train_df['longitude'] = train_df['geohash'].map(lambda x: geo_dict.get(x, (0,0))[1])
train_df['geo_cluster'] = train_df['geohash'].map(geo_cluster_map).fillna(-1).astype(int)

# Temporal features
train_df['day_of_week'] = (train_df['day'] - 1) % 7
train_df['is_weekend'] = (train_df['day_of_week'] >= 5).astype(np.int8)
train_df['is_morning_rush'] = ((train_df['hour'] >= 7) & (train_df['hour'] <= 9)).astype(np.int8)
train_df['is_evening_rush'] = ((train_df['hour'] >= 17) & (train_df['hour'] <= 19)).astype(np.int8)
train_df['is_peak_hour'] = (train_df['is_morning_rush'] | train_df['is_evening_rush']).astype(np.int8)
train_df['is_night'] = ((train_df['hour'] >= 22) | (train_df['hour'] <= 5)).astype(np.int8)
train_df['is_lunch'] = ((train_df['hour'] >= 11) & (train_df['hour'] <= 13)).astype(np.int8)

# Cyclical encodings
train_df['hour_sin'] = np.sin(2 * np.pi * train_df['hour'] / 24)
train_df['hour_cos'] = np.cos(2 * np.pi * train_df['hour'] / 24)
train_df['dow_sin'] = np.sin(2 * np.pi * train_df['day_of_week'] / 7)
train_df['dow_cos'] = np.cos(2 * np.pi * train_df['day_of_week'] / 7)
train_df['bucket_sin'] = np.sin(2 * np.pi * train_df['time_bucket'] / 96)
train_df['bucket_cos'] = np.cos(2 * np.pi * train_df['time_bucket'] / 96)

# Sort for lag computation
train_df.sort_values(['geohash', 'day', 'time_bucket'], inplace=True)
train_df.reset_index(drop=True, inplace=True)
print('Temporal + spatial features done')

## 4. Lag & Rolling Features

In [ ]:
# Lag features
for i in range(1, 6):
    train_df[f'demand_lag_{i}'] = train_df.groupby('geohash')['demand'].shift(i)

train_df['demand_prev_day'] = train_df.groupby('geohash')['demand'].shift(96)
train_df['demand_prev_2day'] = train_df.groupby('geohash')['demand'].shift(192)
train_df['demand_prev_week'] = train_df.groupby('geohash')['demand'].shift(672)

# Demand momentum
train_df['demand_diff_1'] = train_df['demand_lag_1'] - train_df['demand_lag_2']
train_df['demand_diff_2'] = train_df['demand_lag_2'] - train_df['demand_lag_3']

# Rolling statistics
for w in [4, 12, 24, 96]:
    rolled = train_df.groupby('geohash')['demand'].rolling(w, min_periods=1).mean()
    train_df[f'roll_mean_{w}'] = rolled.reset_index(level=0, drop=True).values
    rolled_std = train_df.groupby('geohash')['demand'].rolling(w, min_periods=1).std()
    train_df[f'roll_std_{w}'] = rolled_std.reset_index(level=0, drop=True).values

for w in [12, 96]:
    rolled_max = train_df.groupby('geohash')['demand'].rolling(w, min_periods=1).max()
    train_df[f'roll_max_{w}'] = rolled_max.reset_index(level=0, drop=True).values
    rolled_min = train_df.groupby('geohash')['demand'].rolling(w, min_periods=1).min()
    train_df[f'roll_min_{w}'] = rolled_min.reset_index(level=0, drop=True).values

# EMA
train_df['ema_4'] = train_df.groupby('geohash')['demand'].transform(lambda x: x.ewm(span=4).mean())
train_df['ema_12'] = train_df.groupby('geohash')['demand'].transform(lambda x: x.ewm(span=12).mean())
train_df['ema_96'] = train_df.groupby('geohash')['demand'].transform(lambda x: x.ewm(span=96).mean())

print('Lag + rolling features done')

## 5. Aggregated Statistics

In [ ]:
# Mean demand per geohash
geo_mean = train_df.groupby('geohash')['demand'].mean()
train_df['geo_mean_demand'] = train_df['geohash'].map(geo_mean)
geo_std = train_df.groupby('geohash')['demand'].std()
train_df['geo_std_demand'] = train_df['geohash'].map(geo_std)

# Mean demand per geohash + hour
geo_hour_mean = train_df.groupby(['geohash', 'hour'])['demand'].mean()
train_df['geo_hour_mean'] = train_df.set_index(['geohash', 'hour']).index.map(geo_hour_mean.to_dict().get)
train_df['geo_hour_mean'] = train_df['geo_hour_mean'].fillna(train_df['geo_mean_demand'])

# Mean demand per geohash + time_bucket
geo_bucket_mean = train_df.groupby(['geohash', 'time_bucket'])['demand'].mean()
train_df['geo_bucket_mean'] = train_df.set_index(['geohash', 'time_bucket']).index.map(geo_bucket_mean.to_dict().get)
train_df['geo_bucket_mean'] = train_df['geo_bucket_mean'].fillna(train_df['geo_mean_demand'])

# Mean demand per geohash + day_of_week
geo_dow_mean = train_df.groupby(['geohash', 'day_of_week'])['demand'].mean()
train_df['geo_dow_mean'] = train_df.set_index(['geohash', 'day_of_week']).index.map(geo_dow_mean.to_dict().get)
train_df['geo_dow_mean'] = train_df['geo_dow_mean'].fillna(train_df['geo_mean_demand'])

# Global patterns
hour_mean = train_df.groupby('hour')['demand'].mean()
train_df['hour_mean_global'] = train_df['hour'].map(hour_mean)
bucket_mean = train_df.groupby('time_bucket')['demand'].mean()
train_df['bucket_mean_global'] = train_df['time_bucket'].map(bucket_mean)

# Cluster stats
cluster_mean = train_df.groupby('geo_cluster')['demand'].mean()
train_df['cluster_mean_demand'] = train_df['geo_cluster'].map(cluster_mean)
cluster_hour_mean = train_df.groupby(['geo_cluster', 'hour'])['demand'].mean()
train_df['cluster_hour_mean'] = train_df.set_index(['geo_cluster', 'hour']).index.map(cluster_hour_mean.to_dict().get)
train_df['cluster_hour_mean'] = train_df['cluster_hour_mean'].fillna(train_df['cluster_mean_demand'])

# Neighbor stats
geo_mean_dict = geo_mean.to_dict()
neighbor_mean_demand = {}
for g in all_geohashes:
    neighbors = neighbor_map.get(g, [])
    if neighbors:
        neighbor_mean_demand[g] = np.mean([geo_mean_dict.get(n, 0) for n in neighbors])
    else:
        neighbor_mean_demand[g] = geo_mean_dict.get(g, 0)

train_df['neighbor_mean'] = train_df['geohash'].map(neighbor_mean_demand)
train_df['neighbor_count'] = train_df['geohash'].map(lambda x: len(neighbor_map.get(x, [])))

# Interaction features
train_df['demand_vs_geo'] = train_df['demand_lag_1'] / (train_df['geo_mean_demand'] + 1e-8)
train_df['demand_vs_hour'] = train_df['demand_lag_1'] / (train_df['geo_hour_mean'] + 1e-8)
train_df['peak_geo'] = train_df['is_peak_hour'] * train_df['geo_mean_demand']
train_df['weekend_hour'] = train_df['is_weekend'] * train_df['hour']

print(f'Final training shape: {train_df.shape}')

## 6. Prepare Test Features

In [ ]:
# Feature columns
feature_cols = [
    'latitude', 'longitude', 'geo_cluster',
    'hour', 'minute', 'time_bucket', 'day_of_week',
    'is_weekend', 'is_morning_rush', 'is_evening_rush', 'is_peak_hour',
    'is_night', 'is_lunch',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'bucket_sin', 'bucket_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'demand_lag_4', 'demand_lag_5',
    'demand_prev_day', 'demand_prev_2day', 'demand_prev_week',
    'demand_diff_1', 'demand_diff_2',
    'roll_mean_4', 'roll_mean_12', 'roll_mean_24', 'roll_mean_96',
    'roll_std_4', 'roll_std_12', 'roll_std_24', 'roll_std_96',
    'roll_max_12', 'roll_max_96', 'roll_min_12', 'roll_min_96',
    'ema_4', 'ema_12', 'ema_96',
    'geo_mean_demand', 'geo_std_demand', 'geo_hour_mean', 'geo_bucket_mean',
    'geo_dow_mean', 'hour_mean_global', 'bucket_mean_global',
    'cluster_mean_demand', 'cluster_hour_mean',
    'neighbor_mean', 'neighbor_count',
    'demand_vs_geo', 'demand_vs_hour', 'peak_geo', 'weekend_hour',
]

# Parse test timestamps
test_df['hour'] = test_df['timestamp'].apply(lambda x: int(x.split(':')[0]))
test_df['minute'] = test_df['timestamp'].apply(lambda x: int(x.split(':')[1]))
test_df['time_bucket'] = test_df['hour'] * 4 + test_df['minute'] // 15

# Extract day 49 features from training data
day49_data = train_df[train_df['day'] == 49].copy()
lag_and_agg_cols = [c for c in feature_cols if c not in 
    ['hour', 'minute', 'time_bucket', 'day_of_week', 'is_weekend',
     'is_morning_rush', 'is_evening_rush', 'is_peak_hour', 'is_night', 'is_lunch',
     'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'bucket_sin', 'bucket_cos',
     'latitude', 'longitude', 'geo_cluster']]

day49_lookup = day49_data[['geohash', 'time_bucket'] + lag_and_agg_cols].copy()
day49_lookup = day49_lookup.groupby(['geohash', 'time_bucket']).first().reset_index()

# Merge test with historical features
test_merged = test_df.merge(day49_lookup, on=['geohash', 'time_bucket'], how='left')

# Add spatial/temporal features for test
test_merged['latitude'] = test_merged['geohash'].map(lambda x: geo_dict.get(x, (0,0))[0])
test_merged['longitude'] = test_merged['geohash'].map(lambda x: geo_dict.get(x, (0,0))[1])
test_merged['geo_cluster'] = test_merged['geohash'].map(geo_cluster_map).fillna(0).astype(int)
test_merged['day_of_week'] = (test_merged['day'] - 1) % 7
test_merged['is_weekend'] = (test_merged['day_of_week'] >= 5).astype(int)
test_merged['is_morning_rush'] = ((test_merged['hour'] >= 7) & (test_merged['hour'] <= 9)).astype(int)
test_merged['is_evening_rush'] = ((test_merged['hour'] >= 17) & (test_merged['hour'] <= 19)).astype(int)
test_merged['is_peak_hour'] = (test_merged['is_morning_rush'] | test_merged['is_evening_rush']).astype(int)
test_merged['is_night'] = ((test_merged['hour'] >= 22) | (test_merged['hour'] <= 5)).astype(int)
test_merged['is_lunch'] = ((test_merged['hour'] >= 11) & (test_merged['hour'] <= 13)).astype(int)
test_merged['hour_sin'] = np.sin(2 * np.pi * test_merged['hour'] / 24)
test_merged['hour_cos'] = np.cos(2 * np.pi * test_merged['hour'] / 24)
test_merged['dow_sin'] = np.sin(2 * np.pi * test_merged['day_of_week'] / 7)
test_merged['dow_cos'] = np.cos(2 * np.pi * test_merged['day_of_week'] / 7)
test_merged['bucket_sin'] = np.sin(2 * np.pi * test_merged['time_bucket'] / 96)
test_merged['bucket_cos'] = np.cos(2 * np.pi * test_merged['time_bucket'] / 96)

# Fill missing aggregated features for geohashes not in day49
for col in lag_and_agg_cols:
    if col not in test_merged.columns:
        test_merged[col] = 0
    elif col == 'geo_mean_demand':
        test_merged[col] = test_merged[col].fillna(test_merged['geohash'].map(geo_mean_dict))
    elif col == 'neighbor_mean':
        test_merged[col] = test_merged[col].fillna(test_merged['geohash'].map(neighbor_mean_demand))

test_merged[feature_cols] = test_merged[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
print(f'Test features prepared: {test_merged.shape}')

## 7. Model Training (on entire training.csv)

In [ ]:
# Use days >= 9 (after lag warm-up period)
train_data = train_df[train_df['day'] >= 9].copy()
train_data[feature_cols] = train_data[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)

# Chronological validation: train days 9-47, validate day 48
X_tr = train_data[train_data['day'] <= 47][feature_cols].astype(np.float32)
y_tr = train_data[train_data['day'] <= 47]['demand'].values
X_val = train_data[train_data['day'] == 48][feature_cols].astype(np.float32)
y_val = train_data[train_data['day'] == 48]['demand'].values

print(f'Train: {X_tr.shape[0]:,} rows (days 9-47)')
print(f'Valid: {X_val.shape[0]:,} rows (day 48)')
print(f'Features: {len(feature_cols)}')

In [ ]:
# LightGBM
lgb_model = lgb.LGBMRegressor(
    objective='regression', metric='rmse', boosting_type='gbdt',
    learning_rate=0.03, num_leaves=255, max_depth=-1,
    min_child_samples=50, feature_fraction=0.8, bagging_fraction=0.8,
    bagging_freq=5, reg_alpha=0.1, reg_lambda=0.1,
    n_estimators=2000, verbose=-1, random_state=42, n_jobs=-1,
)
lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)])

lgb_val_pred = np.clip(lgb_model.predict(X_val), 0, 1)
lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_val_pred))
lgb_r2 = r2_score(y_val, lgb_val_pred)
print(f'LightGBM - RMSE: {lgb_rmse:.6f}, R2: {lgb_r2:.6f}, Score: {max(0,100*lgb_r2):.2f}')

In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', learning_rate=0.03, max_depth=8,
    min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, n_estimators=2000,
    random_state=42, tree_method='hist', n_jobs=-1, verbosity=0,
    early_stopping_rounds=50,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)

xgb_val_pred = np.clip(xgb_model.predict(X_val), 0, 1)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_val_pred))
xgb_r2 = r2_score(y_val, xgb_val_pred)
print(f'XGBoost - RMSE: {xgb_rmse:.6f}, R2: {xgb_r2:.6f}, Score: {max(0,100*xgb_r2):.2f}')

In [ ]:
# CatBoost
cat_model = CatBoostRegressor(
    iterations=2000, learning_rate=0.03, depth=8, l2_leaf_reg=3,
    random_seed=42, verbose=200, early_stopping_rounds=50,
    eval_metric='RMSE', task_type='CPU',
)
cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=200)

cat_val_pred = np.clip(cat_model.predict(X_val), 0, 1)
cat_rmse = np.sqrt(mean_squared_error(y_val, cat_val_pred))
cat_r2 = r2_score(y_val, cat_val_pred)
print(f'CatBoost - RMSE: {cat_rmse:.6f}, R2: {cat_r2:.6f}, Score: {max(0,100*cat_r2):.2f}')

## 8. Retrain on All Data + Generate Predictions

In [ ]:
# Retrain on ALL training data (days 9-61) with best iterations from early stopping
X_full = train_data[feature_cols].astype(np.float32)
y_full = train_data['demand'].values

lgb_best = lgb_model.best_iteration_ if lgb_model.best_iteration_ else 1000
xgb_best = xgb_model.best_iteration if xgb_model.best_iteration else 1000
cat_best = cat_model.best_iteration_ if cat_model.best_iteration_ else 1000

print(f'Best iterations - LGB: {lgb_best}, XGB: {xgb_best}, CAT: {cat_best}')

# Final LightGBM
lgb_final = lgb.LGBMRegressor(
    objective='regression', metric='rmse', boosting_type='gbdt',
    learning_rate=0.03, num_leaves=255, max_depth=-1,
    min_child_samples=50, feature_fraction=0.8, bagging_fraction=0.8,
    bagging_freq=5, reg_alpha=0.1, reg_lambda=0.1,
    n_estimators=lgb_best, verbose=-1, random_state=42, n_jobs=-1,
)
lgb_final.fit(X_full, y_full)

# Final XGBoost
xgb_final = xgb.XGBRegressor(
    objective='reg:squarederror', learning_rate=0.03, max_depth=8,
    min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, n_estimators=xgb_best,
    random_state=42, tree_method='hist', n_jobs=-1, verbosity=0,
)
xgb_final.fit(X_full, y_full)

# Final CatBoost
cat_final = CatBoostRegressor(
    iterations=cat_best, learning_rate=0.03, depth=8, l2_leaf_reg=3,
    random_seed=42, verbose=0, eval_metric='RMSE', task_type='CPU',
)
cat_final.fit(X_full, y_full)

print('Final models trained on all data')

## 9. Ensemble & Submission

In [ ]:
# Test predictions
X_test_final = test_merged[feature_cols].astype(np.float32)
X_test_final = X_test_final.replace([np.inf, -np.inf], 0).fillna(0)

lgb_test_pred = np.clip(lgb_final.predict(X_test_final), 0, 1)
xgb_test_pred = np.clip(xgb_final.predict(X_test_final), 0, 1)
cat_test_pred = np.clip(cat_final.predict(X_test_final), 0, 1)

# Optimize ensemble weights on validation set
best_r2 = -999
best_w = (0.33, 0.33, 0.34)
for w1 in np.arange(0.2, 0.7, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        w3 = 1 - w1 - w2
        if w3 < 0.05 or w3 > 0.7:
            continue
        pred = w1 * lgb_val_pred + w2 * xgb_val_pred + w3 * cat_val_pred
        r2 = r2_score(y_val, np.clip(pred, 0, 1))
        if r2 > best_r2:
            best_r2 = r2
            best_w = (w1, w2, w3)

print(f'Optimal weights - LGB: {best_w[0]:.2f}, XGB: {best_w[1]:.2f}, CAT: {best_w[2]:.2f}')
print(f'Ensemble Val R2: {best_r2:.6f} (Score: {max(0, 100*best_r2):.2f})')

# Final prediction
final_pred = best_w[0] * lgb_test_pred + best_w[1] * xgb_test_pred + best_w[2] * cat_test_pred
final_pred = np.clip(final_pred, 0, 1)

# Create submission
submission = pd.DataFrame({'Index': test_df['Index'].values, 'demand': final_pred})
submission.to_csv('submission.csv', index=False)

print(f'\nSubmission saved: {submission.shape[0]} rows')
print(f'Demand stats: mean={final_pred.mean():.4f}, std={final_pred.std():.4f}')
print(submission.head(10))